# Compare Outputs Between Base and Fine-Tuned Models
Uses Unsloth to load a fine-tuned version of Qwen 3.5 0.8B trained for binary query classification as EchoBot's guardian model.

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

import os
os.environ["HF_HUB_OFFLINE"] = "1"

from dotenv import load_dotenv
load_dotenv()
SYSTEM_PROMPT = os.getenv("SYSTEM_PROMPT") 

max_seq_length = 2048

# Load Dataset

In [ ]:
def formatting_func(sample):
    return {
        "prompt": (
            f"{SYSTEM_PROMPT}\n"
            f"User query: {sample['query']}\n"
            "Classification: "
        ),
        "completion": str(sample['label'])
    }

In [10]:
train_path = "../data/overfit/train.jsonl"
val_path = "../data/echobot/val.jsonl"
test_path = "../data/overfit/train.jsonl"

dataset = load_dataset(
    "json",
    data_files={
        "train": train_path,
        "validation": val_path,
        "test": test_path,
    },
)
print(dataset['train'].features)

{'id': Value('string'), 'query': Value('string'), 'label': Value('int64'), 'source': Value('string'), 'tag': Value('string')}


In [11]:
from collections import Counter

# format dataset with only 'prompt' and 'completion' collumns
formatted_train = dataset["train"].map(
    formatting_func,
    remove_columns=dataset["train"].column_names
    )
formatted_val = dataset["validation"].map(
    formatting_func,
    remove_columns=dataset["validation"].column_names
    )
formatted_test = dataset["test"].map(
    formatting_func,
    remove_columns=dataset["test"].column_names
    )

print(Counter(formatted_train["completion"]))
print(Counter(formatted_val["completion"]))
print(Counter(formatted_test["completion"]))


Counter({'0': 100, '1': 100})
Counter({'0': 4, '1': 4})
Counter({'0': 100, '1': 100})


# Compare Models

In [12]:
from evaluation import evaluate_model

In [ ]:
base_model_name = "eb-base-qwen-v1"
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-0.8B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    full_finetuning = False,
    dtype = torch.bfloat16,
    device_map= "auto",
)
FastLanguageModel.for_inference(base_model)

==((====))==  Unsloth 2026.8.12: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear4bit(in_features=768, out_features=2304, bias=True)
            (proj): Linear4bit(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear4bit(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
        )
  

In [ ]:
evaluate_model(
    model=base_model, 
    tokenizer=base_tokenizer,
    dataset=dataset['test'],
    model_str=base_model_name,
    batch_size=16
)

Accuracy: 100/200 = 0.5000
TPR (recall): 0.0000
FPR: 0.0000
Precision: 0.0000


In [15]:
del base_model
del base_tokenizer

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [17]:
lora_model_name = "eb-ovrft-lora-qwen-v1"
lora_model, lora_tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"../adapters/{lora_model_name}",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    full_finetuning = False,
    dtype = torch.bfloat16,
    device_map= "auto",
)
FastLanguageModel.for_inference(lora_model)

==((====))==  Unsloth 2026.8.12: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 768)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-11): 12 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear4bit(in_features=768, out_features=2304, bias=True)
                (proj): Linear4bit(in_features=768, out_features=768, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
           

In [18]:
evaluate_model(
    model=lora_model, 
    tokenizer=lora_tokenizer,
    dataset=dataset['test'],
    model_str=lora_model_name,
    batch_size=16
)

Accuracy: 200/200 = 1.0000
TPR (recall): 1.0000
FPR: 0.0000
Precision: 1.0000
